# 04.04 — Multi-Shape Relationships

In a real graph database the same relationship label can connect different
node types. A `KNOWS` edge might exist between two `Person` nodes **and**
between two `Company` nodes. Orthograph treats these as **distinct shapes**,
each identified by an *identity triple*:

```
Person:KNOWS:Person
Company:KNOWS:Company
```

This notebook shows the full vertical slice for this feature:

1. **Declare** a `GraphDefinition` with two `RelationshipModel` subclasses
   sharing the same `__label__` but different endpoint pairs.
2. **Build a profile by hand** that represents what a matching database looks like.
3. **Validate** the hand-built profile against the definition — expect a clean pass.
4. **Introduce mismatches** (wrong endpoint, missing shape) and observe the errors.
5. **Connect to Neo4j**, seed the two shapes, extract a live profile, and validate it.

## 1. Declare the model

Two node types (`Person`, `Company`) and two relationship types that share
the bare label `KNOWS` but connect different endpoint pairs.

| Triple key | Source | Label | Target |
|---|---|---|---|
| `Person:KNOWS:Person` | Person | KNOWS | Person |
| `Company:KNOWS:Company` | Company | KNOWS | Company |

The `GraphDefinition` duplicate-guard is on the full triple, so two models
with the same `__label__` but different endpoints are explicitly allowed.

In [ ]:
from orthograph.definition import GraphDefinition, NodeModel, RelationshipModel


class Person(NodeModel):
    __label__ = "Person"
    __uid_field__ = "name"
    name: str


class Company(NodeModel):
    __label__ = "Company"
    __uid_field__ = "name"
    name: str


class KnowsPerson(RelationshipModel):
    """A person knows another person."""

    __label__ = "KNOWS"
    __source_label__ = "Person"
    __target_label__ = "Person"
    weight: int


class KnowsCompany(RelationshipModel):
    """A company knows another company."""

    __label__ = "KNOWS"
    __source_label__ = "Company"
    __target_label__ = "Company"
    weight: int


model = GraphDefinition(
    name="SocialNetwork",
    node_types=[Person, Company],
    relationship_types=[KnowsPerson, KnowsCompany],
)

print("Model:", model.name)
print("Node labels:         ", model.node_labels)
print("Relationship labels: ", model.relationship_labels)
print("Relationship keys:   ", model.relationship_keys)

In [ ]:
# relationship_labels collapses both shapes to the bare label.
assert model.relationship_labels == {"KNOWS"}, model.relationship_labels
# relationship_keys exposes the two distinct identity triples used by the comparison engine.
assert model.relationship_keys == {"Person:KNOWS:Person", "Company:KNOWS:Company"}, (
    model.relationship_keys
)
print("OK — both triple keys present, bare label set has one entry")

## 2. Look up a shape by its triple

`get_relationship_type(source, label, target)` returns the declared model for
a specific shape — `None` when the triple does not exist in the definition.

In [ ]:
pp = model.get_relationship_type("Person", "KNOWS", "Person")
cc = model.get_relationship_type("Company", "KNOWS", "Company")
pc = model.get_relationship_type("Person", "KNOWS", "Company")  # not declared

print("Person:KNOWS:Person   ->", pp.__name__ if pp else None)
print("Company:KNOWS:Company ->", cc.__name__ if cc else None)
print("Person:KNOWS:Company  ->", pc)  # None — not in the definition

assert pp is KnowsPerson
assert cc is KnowsCompany
assert pc is None, "undeclared triple must return None"
print(
    "OK — triple-key lookup resolves to correct model class; undeclared triple returns None"
)

## 3. Build a matching profile by hand

A `GraphProfile` stores relationship profiles keyed by the identity triple
string (e.g. `"Person:KNOWS:Person"`). The comparison engine matches each
declared triple key against the corresponding profile entry — never by bare
label alone.

`PropertyProfile.constraint_required` defaults to `None`, meaning *constraint
information is not available* — correct for a hand-built profile with no DB
backing. The engine reports this honestly as `CONSTRAINT_UNVERIFIABLE` at
`INFO` severity, which does **not** make the result invalid.

In [1]:
from orthograph.profile import (
    CardinalityStats,
    GraphProfile,
    NodeTypeProfile,
    PropertyProfile,
    RelationshipTypeProfile,
)


# A profile that satisfies the definition: both shapes present, weight property
# present on every edge of each shape.
valid_profile = GraphProfile(
    source="hand-built",
    node_type_profiles={
        "Person": NodeTypeProfile(
            label="Person",
            count=3,
            property_profiles={
                "name": PropertyProfile(name="name", present_count=3, total_count=3),
            },
        ),
        "Company": NodeTypeProfile(
            label="Company",
            count=2,
            property_profiles={
                "name": PropertyProfile(name="name", present_count=2, total_count=2),
            },
        ),
    },
    rel_type_profiles={
        # Shape 1 — three Person-KNOWS->Person edges
        "Person:KNOWS:Person": RelationshipTypeProfile(
            rel_type="KNOWS",
            source_label="Person",
            target_label="Person",
            count=3,
            property_profiles={
                "weight": PropertyProfile(
                    name="weight", present_count=3, total_count=3
                ),
            },
            cardinality_stats=CardinalityStats(min=1, max=2, mean=1.5, count=2),
        ),
        # Shape 2 — one Company-KNOWS->Company edge
        "Company:KNOWS:Company": RelationshipTypeProfile(
            rel_type="KNOWS",
            source_label="Company",
            target_label="Company",
            count=1,
            property_profiles={
                "weight": PropertyProfile(
                    name="weight", present_count=1, total_count=1
                ),
            },
            cardinality_stats=CardinalityStats(min=1, max=1, mean=1.0, count=1),
        ),
    },
)

assert set(valid_profile.rel_type_profiles) == {
    "Person:KNOWS:Person",
    "Company:KNOWS:Company",
}
print("Profile relationship keys:", set(valid_profile.rel_type_profiles.keys()))
print("OK — both identity triples present in profile")

Profile relationship keys: {'Company:KNOWS:Company', 'Person:KNOWS:Person'}
OK — both identity triples present in profile


## 4. Validate — clean pass

Both declared triples have a matching profile entry, all required properties
are present, and no undeclared shapes exist. The result must be valid.

The `INFO`-level `CONSTRAINT_UNVERIFIABLE` issues are expected: the
hand-built `PropertyProfile` instances have `constraint_required=None`
(the default), so the engine correctly reports it cannot verify DB constraints
— but this is advisory only and does not affect `is_valid`.

In [ ]:
from orthograph.compare import profile_to_definition


result_ok = profile_to_definition(valid_profile, model)

print("is_valid:", result_ok.is_valid)
print("Errors:  ", len(result_ok.errors))
print("Warnings:", len(result_ok.warnings))
print("Info:    ", len([i for i in result_ok.issues if i.severity.value == "info"]))
for issue in result_ok.issues:
    print(f"  [{issue.severity.value}] [{issue.code}] {issue.message}")

assert result_ok.is_valid, result_ok.errors
assert len(result_ok.errors) == 0
assert len(result_ok.warnings) == 0
# INFO issues are advisory: CONSTRAINT_UNVERIFIABLE fires for each required
# property because the hand-built profile carries no constraint information.
info_codes = {i.code for i in result_ok.issues if i.severity.value == "info"}
assert info_codes == {"CONSTRAINT_UNVERIFIABLE"}, info_codes
print(
    "OK — valid, zero errors, zero warnings; INFO-only constraint advisory as expected"
)

## 5. Introduce a mismatch — wrong endpoint

Replace the `Company:KNOWS:Company` profile entry with a
`Person:KNOWS:Company` entry instead. The definition has no declared
`Person:KNOWS:Company` triple, so:

- `Company:KNOWS:Company` is **declared but absent** from the profile → error.
- `Person:KNOWS:Company` is **present in the profile but not declared** → warning.

This proves the comparison engine uses the full triple key and cannot be
fooled by a shape that merely shares the bare label `KNOWS`.

In [ ]:
wrong_endpoint_profile = GraphProfile(
    source="hand-built",
    node_type_profiles=valid_profile.node_type_profiles,
    rel_type_profiles={
        "Person:KNOWS:Person": valid_profile.rel_type_profiles["Person:KNOWS:Person"],
        # Wrong endpoint: Person->Company instead of Company->Company
        "Person:KNOWS:Company": RelationshipTypeProfile(
            rel_type="KNOWS",
            source_label="Person",
            target_label="Company",
            count=1,
            property_profiles={
                "weight": PropertyProfile(
                    name="weight", present_count=1, total_count=1
                ),
            },
        ),
    },
)

result_wrong = profile_to_definition(wrong_endpoint_profile, model)

print("is_valid:", result_wrong.is_valid)
print("Errors:  ", len(result_wrong.errors))
for issue in result_wrong.issues:
    print(f"  [{issue.severity.value}] [{issue.code}] {issue.entity_id}")

assert not result_wrong.is_valid
error_codes = {e.code for e in result_wrong.errors}
assert "MISSING_REL_TYPE" in error_codes, error_codes
missing_ids = {e.entity_id for e in result_wrong.errors if e.code == "MISSING_REL_TYPE"}
assert "Company:KNOWS:Company" in missing_ids, missing_ids
warning_codes = {w.code for w in result_wrong.warnings}
assert "UNEXPECTED_REL_TYPE" in warning_codes, warning_codes
print(
    "OK — MISSING_REL_TYPE for Company:KNOWS:Company; UNEXPECTED_REL_TYPE for Person:KNOWS:Company"
)

## 6. Introduce a mismatch — one shape entirely missing

Drop the `Company:KNOWS:Company` entry from the profile completely.
The definition declares it, so its absence must surface as an error.

In [ ]:
missing_shape_profile = GraphProfile(
    source="hand-built",
    node_type_profiles=valid_profile.node_type_profiles,
    rel_type_profiles={
        # Only the Person shape — Company shape is absent
        "Person:KNOWS:Person": valid_profile.rel_type_profiles["Person:KNOWS:Person"],
    },
)

result_missing = profile_to_definition(missing_shape_profile, model)

print("is_valid:", result_missing.is_valid)
print("Errors:  ", len(result_missing.errors))
for issue in result_missing.errors:
    print(f"  [{issue.code}] {issue.entity_id}  -  {issue.message}")

assert not result_missing.is_valid
assert len(result_missing.errors) == 1
assert result_missing.errors[0].code == "MISSING_REL_TYPE"
assert result_missing.errors[0].entity_id == "Company:KNOWS:Company"
print("OK — exactly one error: MISSING_REL_TYPE for Company:KNOWS:Company")

## 7. Live Neo4j profile extraction

Connect to a real Neo4j instance, seed the same two-shapes dataset, extract
a profile with the inspector, and validate it against the definition.

The inspector issues an `InspectEndpointLabelsQuery` for each bare
relationship type, then builds one `RelationshipTypeProfile` per discovered
`(source_label, rel_type, target_label)` shape — so the two KNOWS shapes are
automatically separated without any extra configuration.

**Requirements**: A running Neo4j instance (default `bolt://localhost:7687`,
password read from `.env` key `NEO4J_PASSWORD`).

> **Warning**: the next cells will **erase all data** in the target database
> (`MATCH (n) DETACH DELETE n`) before seeding the demo dataset, and again
> during cleanup. Run only against a dedicated development or test instance.

In [ ]:
from neo4j import GraphDatabase
from shared.utils import load_env, print_apoc_status


neo4j_uri = load_env("NEO4J_URI", "bolt://localhost:7687")
neo4j_user = load_env("NEO4J_USER", "neo4j")
neo4j_password = load_env("NEO4J_PASSWORD", "password")

print(f"Connecting to {neo4j_uri} ...")
driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_user, neo4j_password))
print("Connected")
print_apoc_status(driver)

In [ ]:
# Wipe any leftover data, then seed the two KNOWS shapes.
driver.execute_query("MATCH (n) DETACH DELETE n")

driver.execute_query(
    "MERGE (p1:Person {name: 'Alice'})"
    " MERGE (p2:Person {name: 'Bob'})"
    " MERGE (p3:Person {name: 'Cara'})"
    " MERGE (c1:Company {name: 'Acme'})"
    " MERGE (c2:Company {name: 'Globex'})"
    " MERGE (p1)-[:KNOWS {weight: 1}]->(p2)"
    " MERGE (p1)-[:KNOWS {weight: 2}]->(p3)"
    " MERGE (p2)-[:KNOWS {weight: 3}]->(p3)"
    " MERGE (c1)-[:KNOWS {weight: 9}]->(c2)"
)

node_records, _, _ = driver.execute_query("MATCH (n) RETURN count(n) AS nodes")
rel_records, _, _ = driver.execute_query("MATCH ()-[r]->() RETURN count(r) AS rels")
node_count = node_records[0]["nodes"]
rel_count = rel_records[0]["rels"]
print(f"Seeded: {node_count} nodes, {rel_count} relationships")

assert node_count == 5, node_count
assert rel_count == 4, rel_count
print("OK — correct seed counts")

In [ ]:
from orthograph.profile import inspect_neo4j


live_profile = inspect_neo4j(driver)

print("Relationship keys in live profile:")
for key in sorted(live_profile.rel_type_profiles):
    rtp = live_profile.rel_type_profiles[key]
    print(f"  {key:<30}  count={rtp.count}")

# Inspector must produce two distinct triple-keyed entries — no blended KNOWS key.
assert "Person:KNOWS:Person" in live_profile.rel_type_profiles
assert "Company:KNOWS:Company" in live_profile.rel_type_profiles
assert "KNOWS" not in live_profile.rel_type_profiles, (
    "bare label must not appear as a key"
)
# Per-shape counts must not be blended.
assert live_profile.rel_type_profiles["Person:KNOWS:Person"].count == 3
assert live_profile.rel_type_profiles["Company:KNOWS:Company"].count == 1
print("OK — two distinct profiles, counts unblended")

In [ ]:
# Inspect each shape's property profile and cardinality.
for key in sorted(live_profile.rel_type_profiles):
    rtp = live_profile.rel_type_profiles[key]
    cs = rtp.cardinality_stats
    print(f"=== {key} ===")
    print(f"  count : {rtp.count}")
    if cs:
        print(f"  degree: min={cs.min}, max={cs.max}, avg={cs.mean:.2f}")
    for prop, pp in rtp.property_profiles.items():
        print(
            f"  {prop:<12} present={pp.present_count}/{pp.total_count}  "
            f"types={pp.observed_types or '(APOC required)'}"
        )
    print()

# Property completeness must be per-shape, not summed.
pp_weight = live_profile.rel_type_profiles["Person:KNOWS:Person"].property_profiles[
    "weight"
]
cc_weight = live_profile.rel_type_profiles["Company:KNOWS:Company"].property_profiles[
    "weight"
]
assert pp_weight.present_count == 3, pp_weight.present_count
assert cc_weight.present_count == 1, cc_weight.present_count
# Cardinality bounds must be per-shape.
pp_cs = live_profile.rel_type_profiles["Person:KNOWS:Person"].cardinality_stats
cc_cs = live_profile.rel_type_profiles["Company:KNOWS:Company"].cardinality_stats
assert pp_cs is not None and pp_cs.max == 2, pp_cs
assert cc_cs is not None and cc_cs.max == 1, cc_cs
print("OK — property completeness and cardinality are unblended per shape")

## 8. Validate the live profile against the definition

The live profile is backed by a real DB, so constraints are inspectable.
We did not create uniqueness constraints in this ad-hoc seed, so required
properties will receive `PROPERTY_UNCONSTRAINED` warnings (advisory — not
errors). `is_valid` must still be `True` because no shape is missing and
all required properties are present on every edge.

In [ ]:
from orthograph.compare import profile_to_definition


result_live = profile_to_definition(inspect_neo4j(driver), model)

print("is_valid:", result_live.is_valid)
print("Errors:  ", len(result_live.errors))
print("Warnings:", len(result_live.warnings))
for issue in result_live.issues:
    icon = (
        "[error]  " if issue.severity.value == "error" else f"[{issue.severity.value}]"
    )
    print(f"  {icon} [{issue.code}] {issue.entity_id}")

assert result_live.is_valid, [str(e) for e in result_live.errors]
assert len(result_live.errors) == 0
# Warnings are advisory — no error-level issues means validation passes.
# PROPERTY_UNCONSTRAINED is expected: we seeded without CREATE CONSTRAINT.
warning_codes = {w.code for w in result_live.warnings}
assert warning_codes <= {"PROPERTY_UNCONSTRAINED", "UNEXPECTED_NODE_LABEL"}, (
    warning_codes
)
print("OK — is_valid=True, zero errors; advisory warnings only")

## 9. Cleanup

In [ ]:
driver.execute_query("MATCH (n) DETACH DELETE n")
driver.close()
print("Database cleared and driver closed.")